In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import LongType, DoubleType
from pyspark.sql.functions import broadcast
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
 
import numpy as np
import json
 
# ── CONFIG ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = "s3://university-research-s20426"
OUTPUT_ROOT = f"{DATA_ROOT}/tgnn_dataset_v2"
 
TRAIN_END = 650
VAL_END   = 700
 
# Robust scaling fallback IQR floor
IQR_FLOOR = 1e-8
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## Part 1 — Load Raw Tables

### Data Loading

In [0]:
# cg  = spark.read.parquet(f"{DATA_ROOT}/MSCallGraph_clean")
# res = spark.read.parquet(f"{DATA_ROOT}/resource")
# rtq = spark.read.parquet(f"{DATA_ROOT}/MSRTQps_clean")
 
# # Add minute-level time index (timestamp is in milliseconds)
# cg  = cg.withColumn("t_idx",  (F.col("timestamp") / 60000).cast("int"))
# res = res.withColumn("t_idx", (F.col("timestamp") / 60000).cast("int"))
# rtq = rtq.withColumn("t_idx", (F.col("timestamp") / 60000).cast("int"))

In [0]:
print("res schema:"); res.printSchema()
print("rtq schema:"); rtq.printSchema()
print("cg  schema:"); cg.printSchema()

res schema:
root
 |-- msname: string (nullable = true)
 |-- msinstanceid: string (nullable = true)
 |-- nodeid: string (nullable = true)
 |-- cpu_utilization: double (nullable = true)
 |-- memory_utilization: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- t_idx: integer (nullable = true)

rtq schema:
root
 |-- timestamp: long (nullable = true)
 |-- msname: string (nullable = true)
 |-- msinstanceid: string (nullable = true)
 |-- t_idx: integer (nullable = true)
 |-- HTTP_MCR: double (nullable = true)
 |-- HTTP_RT: double (nullable = true)
 |-- consumerMQ_MCR: double (nullable = true)
 |-- consumerMQ_RT: double (nullable = true)
 |-- consumerRPC_MCR: double (nullable = true)
 |-- consumerRPC_RT: double (nullable = true)
 |-- providerRPC_MCR: double (nullable = true)
 |-- providerRPC_RT: double (nullable = true)

cg  schema:
root
 |-- traceid: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- rpcid: string (nullable = true)
 |-- UM: string (nullab

In [0]:
print("res t_idx range:", res.agg(F.min("t_idx"), F.max("t_idx")).collect()[0])
print("rtq t_idx range:", rtq.agg(F.min("t_idx"), F.max("t_idx")).collect()[0])
print("cg  t_idx range:", cg.agg(F.min("t_idx"),  F.max("t_idx")).collect()[0])

res t_idx range: Row(min(t_idx)=0, max(t_idx)=719)
rtq t_idx range: Row(min(t_idx)=6, max(t_idx)=720)
cg  t_idx range: Row(min(t_idx)=0, max(t_idx)=720)


In [0]:
def temporal_split(df):
    train = df.filter(F.col("t_idx") <  TRAIN_END)
    val   = df.filter((F.col("t_idx") >= TRAIN_END) & (F.col("t_idx") < VAL_END))
    test  = df.filter(F.col("t_idx") >= VAL_END)
    return train, val, test
 
res_train, res_val, res_test = temporal_split(res)
rtq_train, rtq_val, rtq_test = temporal_split(rtq)
cg_train,  cg_val,  cg_test  = temporal_split(cg)
 
print(f"res  — train: {res_train.count():,}  val: {res_val.count():,}  test: {res_test.count():,}")
print(f"rtq  — train: {rtq_train.count():,}  val: {rtq_val.count():,}  test: {rtq_test.count():,}")
print(f"cg   — train: {cg_train.count():,}   val: {cg_val.count():,}   test: {cg_test.count():,}")

res  — train: 125,169,134  val: 9,607,930  test: 3,842,696
rtq  — train: 2,938,755  val: 153  test: 133
cg   — train: 442,647,290   val: 45,160,302   test: 18,447,204


##  Robust Scaler Helpers

In [0]:
def compute_robust_stats(df, col_name):
    """Compute median and IQR. Returns (q2, iqr)."""
    q1, q2, q3 = df.approxQuantile(col_name, [0.25, 0.5, 0.75], 0.01)
    iqr = q3 - q1
    if iqr < IQR_FLOOR:
        iqr = 1.0
    return float(q2), float(iqr)
 
def fit_robust_scaler(df, cols):
    """Fit scaler on df. Returns stats dict."""
    stats = {}
    for col in cols:
        q2, iqr = compute_robust_stats(df, col)
        stats[col] = {"q2": q2, "iqr": iqr}
        print(f"  {col:42s}  median={q2:.4f}  iqr={iqr:.4f}")
    return stats
 
def apply_robust_scaler(df, stats, suffix="_rs"):
    """Apply robust scaling using pre-fitted stats."""
    for col, s in stats.items():
        df = df.withColumn(
            f"{col}{suffix}",
            (F.col(col) - F.lit(s["q2"])) / F.lit(s["iqr"])
        )
    return df

## Global Node Index (msname level)

In [0]:
all_msnames = (
    res_train.select(F.col("msname").alias("node_key"))
    .union(res_val.select(F.col("msname").alias("node_key")))
    .union(res_test.select(F.col("msname").alias("node_key")))
    .union(cg_train.select(F.col("UM").alias("node_key")))
    .union(cg_val.select(F.col("UM").alias("node_key")))
    .union(cg_test.select(F.col("UM").alias("node_key")))
    .union(cg_train.select(F.col("DM").alias("node_key")))
    .union(cg_val.select(F.col("DM").alias("node_key")))
    .union(cg_test.select(F.col("DM").alias("node_key")))
    .distinct()
)
 
node_index = (
    all_msnames
    .orderBy("node_key")
    .withColumn("node_id", F.monotonically_increasing_id().cast(LongType()))
)
 
node_index.write.mode("overwrite").parquet(f"{OUTPUT_ROOT}/node_index")
node_index = spark.read.parquet(f"{OUTPUT_ROOT}/node_index")
 
print(f"Total unique nodes (msname): {node_index.count():,}")
node_index.limit(5).show(truncate=False)

Total unique nodes (msname): 16,475
+----------------------------------------------------------------+-------+
|node_key                                                        |node_id|
+----------------------------------------------------------------+-------+
|(?)                                                             |0      |
|000a146e1697a98392e775be6d501ffdf724b253a518f00755754cc01e244acf|1      |
|000d30090751fef2c879d4babfc4727851b45379a6f7723b1e4d1cf4231f59e8|2      |
|000de13744ed60dbbee42e43e5a1862f2a76155e79d3b64c251d88728580d46c|3      |
|00137b339665d187e0505f82ba68978b6cbf3cd5f690ce6c6a768dd8910d7bef|4      |
+----------------------------------------------------------------+-------+



## Resource Feature Engineering

In [0]:
def aggregate_res_to_msname(res_df):
    return (
        res_df
        .groupBy("msname", "t_idx")
        .agg(
            # CPU
            F.mean("cpu_utilization").alias("cpu_mean"),
            F.max("cpu_utilization").alias("cpu_max"),
            F.percentile_approx("cpu_utilization", 0.95).alias("cpu_p95"),
            F.stddev("cpu_utilization").alias("cpu_std"),
            F.sum(
                F.when(F.col("cpu_utilization") > 0.8, 1).otherwise(0)
            ).alias("cpu_overload_count"),
            # Memory
            F.mean("memory_utilization").alias("memory_mean"),
            F.max("memory_utilization").alias("memory_max"),
            F.percentile_approx("memory_utilization", 0.95).alias("memory_p95"),
            F.stddev("memory_utilization").alias("memory_std"),
            F.sum(
                F.when(F.col("memory_utilization") > 0.8, 1).otherwise(0)
            ).alias("memory_overload_count"),
            # Scale
            F.count("msinstanceid").alias("num_containers"),
        )
        .fillna(0.0)
    )
 
res_ms_train = aggregate_res_to_msname(res_train)
res_ms_val   = aggregate_res_to_msname(res_val)
res_ms_test  = aggregate_res_to_msname(res_test)
 
print("res_ms sample:")
res_ms_train.limit(3).show()

res_ms sample:
+--------------------+-----+-------------------+------------------+-------------------+--------------------+------------------+------------------+------------------+------------------+--------------------+---------------------+--------------+
|              msname|t_idx|           cpu_mean|           cpu_max|            cpu_p95|             cpu_std|cpu_overload_count|       memory_mean|        memory_max|        memory_p95|          memory_std|memory_overload_count|num_containers|
+--------------------+-----+-------------------+------------------+-------------------+--------------------+------------------+------------------+------------------+------------------+--------------------+---------------------+--------------+
|550b471c718408790...|   14| 0.3292608955940605|0.4644583333438883| 0.3787083333280558|0.030579278688370192|                 0|0.6131708087592289|0.6306743621826172|0.6260519027709961|0.009755048410896546|                    0|           348|
|3557add998c8

In [0]:
# Robust scale resource features (fit on train only)
RES_AGG_COLS = [
    "cpu_mean", "cpu_max", "cpu_p95", "cpu_std", "cpu_overload_count",
    "memory_mean", "memory_max", "memory_p95", "memory_std", "memory_overload_count",
    "num_containers",
]
 
print("Fitting resource scaler on train...")
res_scale_stats = fit_robust_scaler(res_ms_train, RES_AGG_COLS)
 
res_ms_train = apply_robust_scaler(res_ms_train, res_scale_stats)
res_ms_val   = apply_robust_scaler(res_ms_val,   res_scale_stats)
res_ms_test  = apply_robust_scaler(res_ms_test,  res_scale_stats)
 
RES_SCALED_COLS = [f"{c}_rs" for c in RES_AGG_COLS]
print("\nResource scaled columns:", RES_SCALED_COLS)

Fitting resource scaler on train...
  cpu_mean                                    median=0.1102  iqr=0.1107
  cpu_max                                     median=0.1424  iqr=0.1500
  cpu_p95                                     median=0.1325  iqr=0.1336
  cpu_std                                     median=0.0126  iqr=0.0137
  cpu_overload_count                          median=0.0000  iqr=1.0000
  memory_mean                                 median=0.6211  iqr=0.1878
  memory_max                                  median=0.6463  iqr=0.1805
  memory_p95                                  median=0.6411  iqr=0.1850
  memory_std                                  median=0.0071  iqr=0.0101
  memory_overload_count                       median=0.0000  iqr=1.0000
  num_containers                              median=44.0000  iqr=114.0000

Resource scaled columns: ['cpu_mean_rs', 'cpu_max_rs', 'cpu_p95_rs', 'cpu_std_rs', 'cpu_overload_count_rs', 'memory_mean_rs', 'memory_max_rs', 'memory_p95_rs', 'memory_

In [0]:
# Rolling temporal features (on scaled resource)
window_5 = (
    Window
    .partitionBy("msname")
    .orderBy("t_idx")
    .rowsBetween(-4, 0)
)
 
lag_window = (
    Window
    .partitionBy("msname")
    .orderBy("t_idx")
)
 
def add_temporal_features(df):
    # CPU — based on cpu_max_rs (most spike-sensitive)
    df = df.withColumn("cpu_rolling_mean", F.avg("cpu_max_rs").over(window_5))
    df = df.withColumn("cpu_rolling_std",  F.stddev("cpu_max_rs").over(window_5))
    df = df.withColumn("_cpu_lag1",        F.lag("cpu_max_rs", 1).over(lag_window))
    df = df.withColumn("cpu_delta",
        F.when(F.col("_cpu_lag1").isNull(), 0.0)
         .otherwise(F.col("cpu_max_rs") - F.col("_cpu_lag1"))
    ).drop("_cpu_lag1")
 
    # Memory — based on memory_max_rs
    df = df.withColumn("memory_rolling_mean", F.avg("memory_max_rs").over(window_5))
    df = df.withColumn("memory_rolling_std",  F.stddev("memory_max_rs").over(window_5))
    df = df.withColumn("_mem_lag1",           F.lag("memory_max_rs", 1).over(lag_window))
    df = df.withColumn("memory_delta",
        F.when(F.col("_mem_lag1").isNull(), 0.0)
         .otherwise(F.col("memory_max_rs") - F.col("_mem_lag1"))
    ).drop("_mem_lag1")
 
    return df.fillna(0.0)
 
res_ms_train = add_temporal_features(res_ms_train)
res_ms_val   = add_temporal_features(res_ms_val)
res_ms_test  = add_temporal_features(res_ms_test)
 
ROLLING_COLS = [
    "cpu_rolling_mean", "cpu_rolling_std", "cpu_delta",
    "memory_rolling_mean", "memory_rolling_std", "memory_delta",
]
 
# COMMAND ----------
 
# Cyclic time encoding
TOTAL_T = 1440
 
def add_time_encoding(df):
    df = df.withColumn("t_sin", F.sin(2 * np.pi * F.col("t_idx") / TOTAL_T))
    df = df.withColumn("t_cos", F.cos(2 * np.pi * F.col("t_idx") / TOTAL_T))
    return df
 
res_ms_train = add_time_encoding(res_ms_train)
res_ms_val   = add_time_encoding(res_ms_val)
res_ms_test  = add_time_encoding(res_ms_test)

##  RTQ Feature Engineering

In [0]:
def aggregate_rtq_to_msname(rtq_df):
    mcr_aggs = [F.sum(c).alias(c) for c in MCR_COLS]
    rt_aggs  = [F.max(c).alias(c) for c in RT_COLS]
    return (
        rtq_df
        .groupBy("msname", "t_idx")
        .agg(*mcr_aggs, *rt_aggs)
        .fillna(0.0)
    )
 
rtq_ms_train = aggregate_rtq_to_msname(rtq_train)
rtq_ms_val   = aggregate_rtq_to_msname(rtq_val)
rtq_ms_test  = aggregate_rtq_to_msname(rtq_test)
 
print("rtq_ms sample:")
rtq_ms_train.limit(3).show()
print(f"rtq_ms rows  train: {rtq_ms_train.count():,}  val: {rtq_ms_val.count():,}  test: {rtq_ms_test.count():,}")

rtq_ms sample:
+--------------------+-----+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+
|              msname|t_idx|          HTTP_MCR|    consumerMQ_MCR|  consumerRPC_MCR|   providerRPC_MCR|          HTTP_RT|     consumerMQ_RT|    consumerRPC_RT|    providerRPC_RT|
+--------------------+-----+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+
|b21c637c2fbf212c5...|  270|377.55217213403813|               0.0|599.8333333333334|13398.716666666669|4.826685006877579|               0.0|30.412087912087912|0.6824022724198566|
|b234d76768ef8c445...|  290|               0.0|394.91311284130757|75.41666666666667|             281.1|              0.0|19.352098259979527| 87.21022727272727|20.333843797856048|
|b2e84be1c751bfdba...|  277|2718.1944553176586| 1530.831462919974|620.6166666666666| 139.4

In [0]:
# Fix RT zeros then log1p transform
def fix_and_log_rtq(df):
    # Replace 0 RT with epsilon (0 means < 1ms, not truly zero)
    for col in RT_COLS:
        df = df.withColumn(col,
            F.when(F.col(col) == 0, 0.001).otherwise(F.col(col))
        )
    # Log1p all RTQ columns (handles heavy-tailed distributions)
    for col in ALL_RTQ_COLS:
        df = df.withColumn(f"{col}_log", F.log1p(F.col(col)))
    return df
 
rtq_ms_train = fix_and_log_rtq(rtq_ms_train)
rtq_ms_val   = fix_and_log_rtq(rtq_ms_val)
rtq_ms_test  = fix_and_log_rtq(rtq_ms_test)
 
RTQ_LOG_COLS = [f"{c}_log" for c in ALL_RTQ_COLS]
 
# COMMAND ----------
 
# Robust scale RTQ features (fit on train only)
print("Fitting RTQ scaler on train...")
rtq_scale_stats = fit_robust_scaler(rtq_ms_train, RTQ_LOG_COLS)

Fitting RTQ scaler on train...
  HTTP_MCR_log                                median=0.0000  iqr=0.9037
  consumerMQ_MCR_log                          median=1.0367  iqr=5.0327
  consumerRPC_MCR_log                         median=4.2314  iqr=6.6722
  providerRPC_MCR_log                         median=4.3959  iqr=6.6236
  HTTP_RT_log                                 median=0.0010  iqr=0.5911
  consumerMQ_RT_log                           median=0.3528  iqr=2.6311
  consumerRPC_RT_log                          median=2.5764  iqr=1.8457
  providerRPC_RT_log                          median=2.5115  iqr=2.6243


In [0]:
rtq_ms_train = apply_robust_scaler(rtq_ms_train, rtq_scale_stats)
rtq_ms_val   = apply_robust_scaler(rtq_ms_val,   rtq_scale_stats)
rtq_ms_test  = apply_robust_scaler(rtq_ms_test,  rtq_scale_stats)
 
RTQ_SCALED_COLS = [f"{c}_rs" for c in RTQ_LOG_COLS]
print("\nRTQ scaled columns:", RTQ_SCALED_COLS)


RTQ scaled columns: ['HTTP_MCR_log_rs', 'consumerMQ_MCR_log_rs', 'consumerRPC_MCR_log_rs', 'providerRPC_MCR_log_rs', 'HTTP_RT_log_rs', 'consumerMQ_RT_log_rs', 'consumerRPC_RT_log_rs', 'providerRPC_RT_log_rs']


## Build Final Node Feature Table

In [0]:
ALL_NODE_FEATURE_COLS = RES_SCALED_COLS + ROLLING_COLS + RTQ_SCALED_COLS + ["t_sin", "t_cos"]
 
print(f"Total node feature dims: {len(ALL_NODE_FEATURE_COLS)}")
for i, c in enumerate(ALL_NODE_FEATURE_COLS, 1):
    print(f"  {i:2d}. {c}")

Total node feature dims: 27
   1. cpu_mean_rs
   2. cpu_max_rs
   3. cpu_p95_rs
   4. cpu_std_rs
   5. cpu_overload_count_rs
   6. memory_mean_rs
   7. memory_max_rs
   8. memory_p95_rs
   9. memory_std_rs
  10. memory_overload_count_rs
  11. num_containers_rs
  12. cpu_rolling_mean
  13. cpu_rolling_std
  14. cpu_delta
  15. memory_rolling_mean
  16. memory_rolling_std
  17. memory_delta
  18. HTTP_MCR_log_rs
  19. consumerMQ_MCR_log_rs
  20. consumerRPC_MCR_log_rs
  21. providerRPC_MCR_log_rs
  22. HTTP_RT_log_rs
  23. consumerMQ_RT_log_rs
  24. consumerRPC_RT_log_rs
  25. providerRPC_RT_log_rs
  26. t_sin
  27. t_cos


In [0]:
def build_node_features(res_ms_df, rtq_ms_df, split_name):
 
    node_lookup = spark.read.parquet(f"{OUTPUT_ROOT}/node_index")
 
    # Join res + RTQ at msname level
    node_feat = (
        res_ms_df.alias("r")
        .join(
            rtq_ms_df.select("msname", "t_idx", *RTQ_SCALED_COLS).alias("q"),
            on=["msname", "t_idx"],
            how="left"
        )
        .fillna(0.0)
    )
 
    # Attach node_id
    node_feat = (
        node_feat
        .join(
            broadcast(
                node_lookup.select(
                    F.col("node_key").alias("msname"),
                    F.col("node_id")
                )
            ),
            on="msname",
            how="left"
        )
        .select(
            "node_id",
            "msname",
            "t_idx",
            *ALL_NODE_FEATURE_COLS,
        )
        .fillna(0.0)
    )
 
    node_feat.write.mode("overwrite").parquet(
        f"{OUTPUT_ROOT}/node_features/{split_name}"
    )
 
    result = spark.read.parquet(f"{OUTPUT_ROOT}/node_features/{split_name}")
    print(f"[{split_name}] node features: {result.count():,} rows  |  {len(result.columns)} cols")
    return result
 
node_feat_train = build_node_features(res_ms_train, rtq_ms_train, "train")
node_feat_val   = build_node_features(res_ms_val,   rtq_ms_val,   "val")
node_feat_test  = build_node_features(res_ms_test,  rtq_ms_test,  "test")

[train] node features: 846,846 rows  |  30 cols
[val] node features: 65,100 rows  |  30 cols
[test] node features: 26,040 rows  |  30 cols


## Edge Feature Engineering

In [0]:
 
def clean_and_aggregate_cg(cg_df):
    return (
        cg_df
        # UM-side only (rt > 0); DM-side records same call with negative rt
        .filter(F.col("rt") > 0)
        # Deduplicate: aggregate multiple calls on same edge per minute
        .groupBy("UM", "DM", "t_idx", "rpctype")
        .agg(
            F.mean("rt").alias("rt_mean"),
            F.max("rt").alias("rt_max"),
            F.stddev("rt").alias("rt_std"),
            F.count("*").alias("call_count"),
        )
        .fillna(0.0)
        # Log transforms
        .withColumn("rt_log_mean",    F.log1p(F.col("rt_mean")))
        .withColumn("rt_log_max",     F.log1p(F.col("rt_max")))
        .withColumn("call_count_log", F.log1p(F.col("call_count")))
    )
 
cg_clean_train = clean_and_aggregate_cg(cg_train)
cg_clean_val   = clean_and_aggregate_cg(cg_val)
cg_clean_test  = clean_and_aggregate_cg(cg_test)
 
print("cg_clean sample:")
cg_clean_train.limit(3).show()
print(f"cg_clean rows  train: {cg_clean_train.count():,}  val: {cg_clean_val.count():,}  test: {cg_clean_test.count():,}")

cg_clean sample:
+--------------------+--------------------+-----+-------+------------------+------+------------------+----------+------------------+------------------+------------------+
|                  UM|                  DM|t_idx|rpctype|           rt_mean|rt_max|            rt_std|call_count|       rt_log_mean|        rt_log_max|    call_count_log|
+--------------------+--------------------+-----+-------+------------------+------+------------------+----------+------------------+------------------+------------------+
|6da93e1b8565f05eb...|9dc23ef9209e77f1c...|  616|     mc| 1.053923541247485|   7.0|0.2637325334959066|      2485|0.7197518864938968|2.0794415416798357| 7.818430272070656|
|4d980b8ae1f1da1a3...|9a9e8613b6d7d1b57...|  527|    rpc|1.7481203007518797|  72.0| 5.057452584263831|       266|1.0109171509680794| 4.290459441148391|  5.58724865840025|
|dc0c230b729e08e4a...|091794afdcf1abbaf...|   70|   http|20.733333333333334|  34.0| 6.669194922892884|        30|3.0788471802644

In [0]:
# Encode rpctype (fit on train only)
rpctype_pipeline = Pipeline(stages=[
    StringIndexer(
        inputCol="rpctype",
        outputCol="rpctype_idx",
        handleInvalid="keep"
    )
])
 
rpctype_model  = rpctype_pipeline.fit(cg_clean_train)
cg_clean_train = rpctype_model.transform(cg_clean_train)
cg_clean_val   = rpctype_model.transform(cg_clean_val)
cg_clean_test  = rpctype_model.transform(cg_clean_test)
 
print("rpctype encoding:")
cg_clean_train.select("rpctype", "rpctype_idx").distinct().orderBy("rpctype_idx").show()

rpctype encoding:
+-----------+-----------+
|    rpctype|rpctype_idx|
+-----------+-----------+
|         db|        0.0|
|         mc|        1.0|
|        rpc|        2.0|
|         mq|        3.0|
|       http|        4.0|
|userDefined|        5.0|
+-----------+-----------+



In [0]:
# Robust scale edge features (fit on train only)
EDGE_LOG_COLS = ["rt_log_mean", "rt_log_max", "rt_std", "call_count_log"]
 
print("Fitting edge scaler on train...")
edge_scale_stats = fit_robust_scaler(cg_clean_train, EDGE_LOG_COLS)
 
cg_clean_train = apply_robust_scaler(cg_clean_train, edge_scale_stats)
cg_clean_val   = apply_robust_scaler(cg_clean_val,   edge_scale_stats)
cg_clean_test  = apply_robust_scaler(cg_clean_test,  edge_scale_stats)
 
EDGE_SCALED_COLS  = [f"{c}_rs" for c in EDGE_LOG_COLS]
EDGE_FEATURE_COLS = EDGE_SCALED_COLS + ["rpctype_idx"]
print("\nEdge feature columns:", EDGE_FEATURE_COLS)

Fitting edge scaler on train...
  rt_log_mean                                 median=0.7732  iqr=0.5306
  rt_log_max                                  median=1.0986  iqr=1.2528
  rt_std                                      median=0.0000  iqr=0.7269
  call_count_log                              median=1.6094  iqr=1.5404

Edge feature columns: ['rt_log_mean_rs', 'rt_log_max_rs', 'rt_std_rs', 'call_count_log_rs', 'rpctype_idx']


In [0]:
def build_edge_features(cg_df, split_name):
 
    node_lookup  = spark.read.parquet(f"{OUTPUT_ROOT}/node_index")
    node_lookup2 = spark.read.parquet(f"{OUTPUT_ROOT}/node_index")
 
    # UM -> src_node_id
    cg_with_src = (
        cg_df
        .join(
            broadcast(node_lookup.select(
                F.col("node_key"),
                F.col("node_id").alias("src_node_id")
            )),
            on=cg_df["UM"] == node_lookup["node_key"],
            how="left"
        )
        .drop("node_key")
    )
 
    # DM -> dst_node_id (fresh read avoids lineage conflicts)
    cg_with_dst = (
        cg_with_src
        .join(
            broadcast(node_lookup2.select(
                F.col("node_key"),
                F.col("node_id").alias("dst_node_id")
            )),
            on=cg_with_src["DM"] == node_lookup2["node_key"],
            how="left"
        )
        .drop("node_key")
    )
 
    edge_feat = (
        cg_with_dst
        .select(
            "src_node_id",
            "dst_node_id",
            "t_idx",
            "UM",
            "DM",
            *EDGE_FEATURE_COLS,
        )
        # Keep edge if at least one endpoint is resolved
        .filter(
            F.col("src_node_id").isNotNull() |
            F.col("dst_node_id").isNotNull()
        )
    )
 
    edge_feat.write.mode("overwrite").parquet(
        f"{OUTPUT_ROOT}/edge_features/{split_name}"
    )
 
    result = spark.read.parquet(f"{OUTPUT_ROOT}/edge_features/{split_name}")
    print(f"[{split_name}] edge features: {result.count():,} rows  |  {len(result.columns)} cols")
    return result
 
edge_feat_train = build_edge_features(cg_clean_train, "train")
edge_feat_val   = build_edge_features(cg_clean_val,   "val")
edge_feat_test  = build_edge_features(cg_clean_test,  "test")
 

[train] edge features: 4,436,440 rows  |  10 cols
[val] edge features: 403,037 rows  |  10 cols
[test] edge features: 156,856 rows  |  10 cols


In [0]:
def write_snapshots(node_feat_df, edge_feat_df, split_name):
 
    node_cols = [c for c in node_feat_df.columns if c != "t_idx"]
    edge_cols = [c for c in edge_feat_df.columns if c != "t_idx"]
 
    (
        node_feat_df
        .select("t_idx", *node_cols)
        .write
        .mode("overwrite")
        .partitionBy("t_idx")
        .option("compression", "snappy")
        .parquet(f"{OUTPUT_ROOT}/snapshots/{split_name}/nodes")
    )
 
    (
        edge_feat_df
        .select("t_idx", *edge_cols)
        .write
        .mode("overwrite")
        .partitionBy("t_idx")
        .option("compression", "snappy")
        .parquet(f"{OUTPUT_ROOT}/snapshots/{split_name}/edges")
    )
 
    node_parts = dbutils.fs.ls(f"{OUTPUT_ROOT}/snapshots/{split_name}/nodes")
    edge_parts = dbutils.fs.ls(f"{OUTPUT_ROOT}/snapshots/{split_name}/edges")
    n_n = sum(1 for p in node_parts if "t_idx=" in p.name)
    n_e = sum(1 for p in edge_parts if "t_idx=" in p.name)
    print(f"[{split_name}]  node partitions: {n_n}   edge partitions: {n_e}")
 
write_snapshots(node_feat_train, edge_feat_train, "train")
write_snapshots(node_feat_val,   edge_feat_val,   "val")
write_snapshots(node_feat_test,  edge_feat_test,  "test")

[train]  node partitions: 650   edge partitions: 650
[val]  node partitions: 50   edge partitions: 50
[test]  node partitions: 20   edge partitions: 21


## Write Manifests

In [0]:
def write_manifest(node_feat_df, split_name):
 
    t_indices = (
        node_feat_df
        .select("t_idx")
        .distinct()
        .orderBy("t_idx")
        .toPandas()["t_idx"]
        .astype(int)
        .tolist()
    )
 
    manifest = {
        "split":      split_name,
        "t_indices":  t_indices,
        "count":      len(t_indices),
        "nodes_path": f"{OUTPUT_ROOT}/snapshots/{split_name}/nodes",
        "edges_path": f"{OUTPUT_ROOT}/snapshots/{split_name}/edges",
    }
 
    mdf = spark.createDataFrame([(json.dumps(manifest, indent=2),)], ["json"])
    mdf.coalesce(1).write.mode("overwrite").text(
        f"{OUTPUT_ROOT}/manifests/{split_name}"
    )
 
    print(f"[{split_name}] manifest: {len(t_indices)} snapshots  "
          f"(t={t_indices[0]} -> t={t_indices[-1]})")
    return t_indices
 
train_t = write_manifest(node_feat_train, "train")
val_t   = write_manifest(node_feat_val,   "val")
test_t  = write_manifest(node_feat_test,  "test")
 

[train] manifest: 650 snapshots  (t=0 -> t=649)
[val] manifest: 50 snapshots  (t=650 -> t=699)
[test] manifest: 20 snapshots  (t=700 -> t=719)


## write Dataset Statistic

In [0]:
stats = {
    "version":             "v2",
    "graph_level":         "msname (microservice)",
    "num_nodes":           node_index.count(),
    "num_node_features":   len(ALL_NODE_FEATURE_COLS),
    "num_edge_features":   len(EDGE_FEATURE_COLS),
    "node_feature_names":  ALL_NODE_FEATURE_COLS,
    "edge_feature_names":  EDGE_FEATURE_COLS,
    "train_t_range":       [int(train_t[0]), int(train_t[-1])],
    "val_t_range":         [int(val_t[0]),   int(val_t[-1])],
    "test_t_range":        [int(test_t[0]),  int(test_t[-1])],
    "s3_root":             OUTPUT_ROOT,
    "feature_engineering": {
        "resource_agg":   "max/p95/std/mean/overload_count per msname per t_idx",
        "mcr_agg":        "sum across containers (total throughput)",
        "rt_agg":         "max across containers (spike-preserving)",
        "edge_rt_filter": "rt > 0 only (UM-side recordings)",
        "edge_dedup":     "group by (UM, DM, t_idx, rpctype)",
        "scaling":        "robust median/IQR, fit on train only",
        "rt_transform":   "log1p before robust scaling",
    },
}
 
stats_df = spark.createDataFrame([(json.dumps(stats, indent=2),)], ["json"])
stats_df.coalesce(1).write.mode("overwrite").text(f"{OUTPUT_ROOT}/dataset_stats")
print(json.dumps(stats, indent=2))

{
  "version": "v2",
  "graph_level": "msname (microservice)",
  "num_nodes": 16475,
  "num_node_features": 27,
  "num_edge_features": 5,
  "node_feature_names": [
    "cpu_mean_rs",
    "cpu_max_rs",
    "cpu_p95_rs",
    "cpu_std_rs",
    "cpu_overload_count_rs",
    "memory_mean_rs",
    "memory_max_rs",
    "memory_p95_rs",
    "memory_std_rs",
    "memory_overload_count_rs",
    "num_containers_rs",
    "cpu_rolling_mean",
    "cpu_rolling_std",
    "cpu_delta",
    "memory_rolling_mean",
    "memory_rolling_std",
    "memory_delta",
    "HTTP_MCR_log_rs",
    "consumerMQ_MCR_log_rs",
    "consumerRPC_MCR_log_rs",
    "providerRPC_MCR_log_rs",
    "HTTP_RT_log_rs",
    "consumerMQ_RT_log_rs",
    "consumerRPC_RT_log_rs",
    "providerRPC_RT_log_rs",
    "t_sin",
    "t_cos"
  ],
  "edge_feature_names": [
    "rt_log_mean_rs",
    "rt_log_max_rs",
    "rt_std_rs",
    "call_count_log_rs",
    "rpctype_idx"
  ],
  "train_t_range": [
    0,
    649
  ],
  "val_t_range": [
    650,
  

## Sanity Check

In [0]:
for split, t_list in [("train", train_t), ("val", val_t), ("test", test_t)]:
 
    t0 = t_list[0]
 
    nodes = (
        spark.read
        .option("mergeSchema", "true")
        .option("basePath", f"{OUTPUT_ROOT}/snapshots/{split}/nodes")
        .parquet(f"{OUTPUT_ROOT}/snapshots/{split}/nodes/t_idx={t0}")
    )
    edges = (
        spark.read
        .option("mergeSchema", "true")
        .option("basePath", f"{OUTPUT_ROOT}/snapshots/{split}/edges")
        .parquet(f"{OUTPUT_ROOT}/snapshots/{split}/edges/t_idx={t0}")
    )
 
    n_nodes = nodes.count()
    n_edges = edges.count()
 
    print(f"\n=== {split.upper()} ===")
    print(f"  Snapshots      : {len(t_list)}  (t={t_list[0]} -> t={t_list[-1]})")
    print(f"  t={t0} nodes   : {n_nodes:,}")
    print(f"  t={t0} edges   : {n_edges:,}")
    print(f"  Node feat cols : {len(nodes.columns)}")
    print(f"  Edge feat cols : {len(edges.columns)}")
 
    if n_nodes == 0: print("  WARNING: zero nodes")
    if n_edges == 0: print("  WARNING: zero edges")


=== TRAIN ===
  Snapshots      : 650  (t=0 -> t=649)
  t=0 nodes   : 1,303
  t=0 edges   : 7,910
  Node feat cols : 30
  Edge feat cols : 10

=== VAL ===
  Snapshots      : 50  (t=650 -> t=699)
  t=650 nodes   : 1,302
  t=650 edges   : 8,419
  Node feat cols : 30
  Edge feat cols : 10

=== TEST ===
  Snapshots      : 20  (t=700 -> t=719)
  t=700 nodes   : 1,302
  t=700 edges   : 7,772
  Node feat cols : 30
  Edge feat cols : 10


## Sample Display

In [0]:
t0 = train_t[0]
 
sample_nodes = (
    spark.read
    .option("mergeSchema", "true")
    .option("basePath", f"{OUTPUT_ROOT}/snapshots/train/nodes")
    .parquet(f"{OUTPUT_ROOT}/snapshots/train/nodes/t_idx={t0}")
)
sample_edges = (
    spark.read
    .option("mergeSchema", "true")
    .option("basePath", f"{OUTPUT_ROOT}/snapshots/train/edges")
    .parquet(f"{OUTPUT_ROOT}/snapshots/train/edges/t_idx={t0}")
)
 
print(f"Node columns ({len(sample_nodes.columns)}):", sample_nodes.columns)
print(f"\nEdge columns ({len(sample_edges.columns)}):", sample_edges.columns)
 
display(sample_nodes.limit(5))
display(sample_edges.limit(5))

Node columns (30): ['node_id', 'msname', 'cpu_mean_rs', 'cpu_max_rs', 'cpu_p95_rs', 'cpu_std_rs', 'cpu_overload_count_rs', 'memory_mean_rs', 'memory_max_rs', 'memory_p95_rs', 'memory_std_rs', 'memory_overload_count_rs', 'num_containers_rs', 'cpu_rolling_mean', 'cpu_rolling_std', 'cpu_delta', 'memory_rolling_mean', 'memory_rolling_std', 'memory_delta', 'HTTP_MCR_log_rs', 'consumerMQ_MCR_log_rs', 'consumerRPC_MCR_log_rs', 'providerRPC_MCR_log_rs', 'HTTP_RT_log_rs', 'consumerMQ_RT_log_rs', 'consumerRPC_RT_log_rs', 'providerRPC_RT_log_rs', 't_sin', 't_cos', 't_idx']

Edge columns (10): ['src_node_id', 'dst_node_id', 'UM', 'DM', 'rt_log_mean_rs', 'rt_log_max_rs', 'rt_std_rs', 'call_count_log_rs', 'rpctype_idx', 't_idx']


node_id,msname,cpu_mean_rs,cpu_max_rs,cpu_p95_rs,cpu_std_rs,cpu_overload_count_rs,memory_mean_rs,memory_max_rs,memory_p95_rs,memory_std_rs,memory_overload_count_rs,num_containers_rs,cpu_rolling_mean,cpu_rolling_std,cpu_delta,memory_rolling_mean,memory_rolling_std,memory_delta,HTTP_MCR_log_rs,consumerMQ_MCR_log_rs,consumerRPC_MCR_log_rs,providerRPC_MCR_log_rs,HTTP_RT_log_rs,consumerMQ_RT_log_rs,consumerRPC_RT_log_rs,providerRPC_RT_log_rs,t_sin,t_cos,t_idx
16362,fe1c1b555b650c8a5c38d93fd204ef573e2d7f851096ef2d9b3cd630a9a77469,-0.18321034792286867,-0.22336435616236408,-0.23573433117057882,-0.3586487487062264,0.0,-0.5989833963344989,-0.4566986966940467,-0.6436572984316951,0.23121894845881283,0.0,2.456140350877193,-0.22336435616236408,0.0,0.0,-0.4566986966940467,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
15936,f74d85cf1b5d10fd64eb74005bfc1209bbe853c2b4a6ac3fb857ceaaf55267d6,0.7834925034004536,0.6562022503170473,0.8013719987451905,1.5510826693686888,0.0,-0.38395578652597906,-0.5028705491956386,-0.4667324961133985,-0.32122895546747443,0.0,-0.21052631578947367,0.6562022503170473,0.0,0.0,-0.5028705491956386,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
15867,f622027832aa340a3117bacfa253d94953e0c15e9eed3a4adc2b0a162f335f8c,0.973549922797239,0.7028753993711487,0.8628001247254393,0.22355843489580193,0.0,0.2578208004865951,0.1732976837455317,0.19717895898378585,-0.35652794307003616,0.0,-0.12280701754385964,0.7028753993711487,0.0,0.0,0.1732976837455317,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
14371,df109bf2a3bda3c6b06e1c7e951edcfa60485111af5c304fdeb3ef3ddc2f9b63,-0.7207415628277648,-0.7330184747908893,-0.7488306828769429,-0.7528104964459652,0.0,0.2170322630946813,0.10990914110737413,0.1353220529505944,-0.4199199364731739,0.0,-0.3157894736842105,-0.7330184747908893,0.0,0.0,0.10990914110737413,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
13876,d7b85389a523bf9cb0b798a4c3ac9ba515cac94998b8948ee6461500439ba3ba,-0.17147132334669424,-0.12779552715956674,-0.1992516370375591,-0.2402489808965916,0.0,0.8295108036201625,0.7700205284558216,0.7747988377579144,1.6479192389272836,0.0,0.9824561403508771,-0.12779552715956674,0.0,0.0,0.7700205284558216,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0


src_node_id,dst_node_id,UM,DM,rt_log_mean_rs,rt_log_max_rs,rt_std_rs,call_count_log_rs,rpctype_idx,t_idx
14214,9577,dccacd558ca56ae655fbfc8914f322b468f071db1fade7285688371cdf6badff,949b5239adb1593786fc7bb4e34ed7ffa792cd7ea85b6ad445ac3aaab7e8c007,1.5759583522488338,0.4077591983577853,0.0,-0.5948220855128786,2.0,0
12409,13385,c023763c547199120799c2c5bffb52047d0f275bf1391f752a02ff074a5a974f,d03bb97862607468fe3153b28d41a20de1e3144a5662642b8d4c1062c550f622,4.1885112239531095,1.5143487096880097,0.0,-0.5948220855128786,2.0,0
11328,10149,af42b5e3e0eb334d38619733586d78d1414f6549f24d31b39a5294454638bc59,9d93c5a28418f327c0697d62dd974cb71cebca697c591e05dacf372a260bd2a5,-0.1508451679934365,-0.32365668390976504,0.0,-0.5948220855128786,0.0,0
11328,9280,af42b5e3e0eb334d38619733586d78d1414f6549f24d31b39a5294454638bc59,9092a2565968a8b2fda9fae87c2d4fb2799cb0c438772f085cc19c606f013c94,-0.1508451679934365,-0.32365668390976504,0.0,-0.5948220855128786,0.0,0
6288,5731,6169acee51601352d50e58aa212e556a58b0fc55d1f54bf49f72cf47e6748472,58fdf9e273b344f8e1a2ece611b1a59bd087a7084bb427b2b3fda134f4e7c2f0,-0.1508451679934365,-0.32365668390976504,0.0,-0.5948220855128786,1.0,0
